[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/fast_track/06_pandas_fundamentals.ipynb)

# 📓 Notebook 6 (fast track) — Pandas Fundamentals: Your First Real DataFrame

> **Module:** Data Science Libraries · **Estimated time:** 30–40 min · **Difficulty:** Beginner

You can do real data work in pure Python — that's exactly what you did with lists (Notebook 3) and dicts (Notebook 4). But once your data is more than a handful of rows, you want a **proper table**: row + column labels, fast filtering, group-by operations, reading and writing CSV files. That is **pandas**.

Pandas is *the* library every data scientist touches every day. This notebook gives you a working preview: enough to be productive, with the deep dive coming after NumPy and matplotlib.

In keeping with the course identity, every example in this notebook uses a **business-AI dataset** — a log of LLM API calls with their model, token usage, cost, latency, and customer segment. By the end of the notebook you'll have built a small DataFrame, filtered it, grouped it, and drawn your first plot from it.

## 🎯 Learning objectives

1. Build a `DataFrame` from a dict and from a CSV.
2. Inspect a new dataset with the standard "first five commands".
3. Select columns and rows (`.loc` vs `.iloc`).
4. Filter rows with **boolean masks** — the pandas idiom.
5. Add derived columns from formulas.
6. Use `groupby` to compute aggregates by category.
7. Draw a first plot directly from a DataFrame.

## ✅ Prerequisites

Notebooks 1–4.

> 🏎️ **You're on the fast track.** This is a trimmed version of the canonical [`02_data_science/07_pandas_fundamentals.ipynb`](../02_data_science/07_pandas_fundamentals.ipynb) — pandas fundamentals. The Stretch exercises (A–D) and the 🎁 Bonus mini-project have been removed to keep the notebook under ~90 minutes. Open the canonical version once you want the deeper material.

---


## 1. The pandas mental model

Pandas has two core objects:

| Object        | Conceptually          | Compare to                        |
|---------------|-----------------------|-----------------------------------|
| `Series`      | A single column       | a labelled NumPy 1-D array        |
| `DataFrame`   | A whole table         | a labelled 2-D array / SQL table  |

```
   ┌────────────────  DataFrame  ────────────────┐
   │  request_id   model       tokens   cost    │
   │  req_001      gpt-4o-mini  482     0.00029 │
   │  req_002      claude-haiku 612     0.00037 │
   │  ...                                       │
   └────────────────────────────────────────────┘
                ↑                  ↑
               row              column
            (a record)        (a Series)
```

If you understand a list-of-dicts from Notebook 4, you already understand a DataFrame — pandas just gives you a *much* better toolkit for working with it.

In [ ]:
# The standard import aliases — you'll see "pd" and "np" in every pandas notebook ever
import pandas as pd
import numpy as np

print(f"pandas version: {pd.__version__}")
print(f"numpy  version: {np.__version__}")


## 2. Creating a DataFrame from a dict

The most natural way to build a small DataFrame is from a dict where **keys are column names** and **values are lists** (one per row).

In [ ]:
# A small log of LLM API calls from yesterday's batch
data = {
    "request_id":     ["req_001", "req_002", "req_003", "req_004", "req_005", "req_006", "req_007", "req_008"],
    "model":          ["gpt-4o-mini", "gpt-4o-mini", "claude-haiku", "gpt-4o-mini",
                       "claude-haiku", "gpt-4o-mini", "claude-haiku", "gpt-4o-mini"],
    "tokens_in":      [482, 612, 510, 1180, 720,  430,  890, 240],
    "tokens_out":     [121, 158, 144,  205, 192,  118,  201,  87],
    "latency_ms":     [1820, 2150, 1640, 3180, 2410, 1740, 2530, 1380],
    "customer_seg":   ["SMB", "Enterprise", "SMB", "Enterprise",
                       "Mid-market", "SMB", "Enterprise", "SMB"],
}

df = pd.DataFrame(data)
df


**A pandas tip.** Inside Jupyter, just typing the variable name (`df`) at the bottom of a cell renders a beautifully formatted HTML table. Try `print(df)` *and* just `df` — the latter is what you'll prefer.

## 3. Inspecting a DataFrame

Whenever you load a new dataset, run these commands first. They tell you the *shape*, the *types*, and the *typical values* before you do anything else. Skipping this step is one of the most common rookie mistakes.

In [ ]:
print("Shape (rows, cols):", df.shape)
print("\nColumn names :", df.columns.tolist())
print("\nData types:")
print(df.dtypes)


In [ ]:
# The first / last few rows
print("--- head(3) ---")
print(df.head(3))

print("\n--- tail(2) ---")
print(df.tail(2))


In [ ]:
# Quick statistical summary for numeric columns
df.describe()


In [ ]:
# Schema and non-null counts — your missing-data canary
df.info()


## 4. Selecting columns

These three patterns appear in every pandas notebook:

```python
df["tokens_in"]                       # one column → Series
df[["request_id", "tokens_in"]]       # several columns → DataFrame
df.tokens_in                          # attribute-style (only when the name is a valid identifier)
```

In [ ]:
# One column → Series
tokens = df["tokens_in"]
print(tokens)
print(f"\ntype: {type(tokens).__name__}")


In [ ]:
# Several columns → DataFrame
subset = df[["request_id", "model", "tokens_in", "tokens_out"]]
subset


## 5. Selecting rows — `loc` vs `iloc`

Pandas gives you **two** ways to slice rows. The difference matters once your DataFrame has a non-default index.

| Indexer | What it uses          | Endpoint inclusive?   |
|---------|------------------------|-----------------------|
| `.loc`  | **labels** (the index) | yes (`df.loc[0:2]` returns 3 rows) |
| `.iloc` | **integer positions**  | no (`df.iloc[0:2]` returns 2 rows) |

In [ ]:
# .loc — label-based
print("df.loc[0]:")
print(df.loc[0])           # first row → Series

print("\ndf.loc[0:2, ['request_id', 'tokens_in']]:")
print(df.loc[0:2, ["request_id", "tokens_in"]])


In [ ]:
# .iloc — position-based
print("df.iloc[0]:")
print(df.iloc[0])

print("\ndf.iloc[0:2, 0:3]:")
print(df.iloc[0:2, 0:3])    # first 2 rows, first 3 cols


> 🎯 **Rule of thumb.** Prefer `.loc` — it reads almost like English. Use `.iloc` only when you really do mean "the *n*-th row" regardless of labels.

### 🔬 What actually happens — `loc` vs `iloc` when the index isn't `0,1,2`

So far `df.loc[0]` and `df.iloc[0]` returned the **same** row — but only by accident: our index *happened* to be `0,1,2,…`, so label `0` and position `0` coincided. The moment the index is something else (after filtering, after `set_index`, after sorting), they diverge. Here is the mental model:

- **`.loc` is LABEL-based** — it looks up the value *printed in the index*. `df.loc["b"]` means "the row whose index label is `b`".
- **`.iloc` is POSITION-based** — it counts from the top, `0, 1, 2, …`, ignoring whatever the labels say. `df.iloc[0]` always means "the *first* row".

Picture a tiny frame with a string index:

```text
         index  │  val          .loc["b"]  → label "b" → val 20   (2nd row here, but that's incidental)
        ────────┼──────         .iloc[0]   → position 0 → val 10   (always the FIRST row)
   pos 0 →   a  │  10           .iloc[-1]  → last position → val 30
   pos 1 →   b  │  20
   pos 2 →   c  │  30           df.loc[0]  → KeyError!  there is no LABEL 0 anymore
```

The trap: after you filter a DataFrame, pandas **keeps the original labels**, so the surviving rows have a gappy index like `1, 3, 4`. Now `df.iloc[0]` is the first surviving row, but `df.loc[0]` raises `KeyError` because label `0` was filtered away.


In [ ]:
# PROOF — index is NOT 0,1,2, so .loc and .iloc point at different rows
mini = pd.DataFrame({"val": [10, 20, 30]}, index=["a", "b", "c"])
print(mini, "\n")

print("iloc[0]  (position 0, the FIRST row):", mini.iloc[0]["val"])   # 10
print("loc['b'] (label 'b'):                ", mini.loc["b"]["val"])  # 20
print("iloc[1]  (position 1) == loc['b']?   ", mini.iloc[1]["val"] == mini.loc["b"]["val"])  # True

# The classic filtering trap: surviving rows keep their ORIGINAL labels.
left = df[df["tokens_in"] > 600]      # keeps a gappy index, e.g. 1, 3, 4, 6
print("\nIndex left after filtering tokens_in > 600:", left.index.tolist())
print("iloc[0] request_id (first surviving row):", left.iloc[0]["request_id"])
try:
    left.loc[0]                       # label 0 was filtered out → KeyError
except KeyError:
    print("loc[0] -> KeyError: label 0 no longer exists (iloc[0] still works!)")


### 🧠 `loc` vs `iloc` vs boolean mask — one table

| You write | Selects by | `mini.…[0]` on index `['a','b','c']` |
|---|---|---|
| `df.loc[lbl]` | **label** (index value) | `mini.loc[0]` → `KeyError` (no label `0`) |
| `df.iloc[pos]` | **integer position** | `mini.iloc[0]` → first row (`val=10`) |
| `df[df.col > x]` | a **boolean Series** | keeps rows where the mask is `True` |

**Boolean masking is the same idea, one level up.** `df["tokens_in"] > 500` doesn't return rows — it returns a `True`/`False` **Series the same length as `df`**. Indexing with that Series keeps only the `True` rows. Combine masks with **`&`** (and) / **`|`** (or), and **every condition needs parentheses** because `&`/`|` bind tighter than `>`/`==`:

```python
df[(df["tokens_in"] > 500) & (df["latency_ms"] > 2000)]   # ✅ parens required
df[ df["tokens_in"] > 500  &  df["latency_ms"] > 2000 ]   # ❌ ValueError / wrong result
```

> ⚠️ **Two traps in one.** (1) Use `&`/`|`, *never* Python's `and`/`or`, on Series — `and` tries to collapse the whole Series to one truth value and raises `ValueError`. (2) After filtering, reach for `.iloc` (position) or `.reset_index(drop=True)` if you want clean `0,1,2…` labels back — `.loc` with an old integer label will surprise you.

> 🎯 **Rule of thumb (refined).** Use `.loc` when you mean *"the row called X"* and `.iloc` when you mean *"the n-th row"*. They only look interchangeable while the index is the default `0,1,2,…` — which is exactly when bugs hide.


## 6. Filtering — the boolean-mask idiom

This is **the** pattern of pandas: build a Series of `True`/`False` values that says which rows you want, then index the DataFrame with it.

```python
df[ df["tokens_in"] > 500 ]
   └────────────────────┘
   boolean Series the same length as df
```

In [ ]:
# Step 1: build the mask
mask = df["tokens_in"] > 500
print("Mask:")
print(mask)

# Step 2: apply it
print("\nLarge-prompt requests:")
print(df[mask])


In [ ]:
# Combining conditions — note the parentheses around each part, and & (not `and`)
big_and_slow = df[(df["tokens_in"] > 500) & (df["latency_ms"] > 2000)]
print("Big AND slow requests:")
print(big_and_slow)

# OR uses |
enterprise_or_huge = df[(df["customer_seg"] == "Enterprise") | (df["tokens_in"] > 1000)]
print("\nEnterprise OR very long:")
print(enterprise_or_huge)


> ⚠️ Two common stumbles:
> - Use **`&` and `|`**, *not* `and` / `or`, for element-wise boolean operations.
> - **Wrap each condition in parentheses** — `&` and `|` have higher precedence than `>` and `==`.

## 7. Adding and modifying columns

Derived columns are how you turn raw data into business KPIs.

In [ ]:
# A simple price table (you'd normally load this from config)
prices_per_1k = {
    "gpt-4o-mini" : {"in": 0.0006, "out": 0.0024},
    "claude-haiku": {"in": 0.0008, "out": 0.0040},
}

# Map model → price using a small lambda
df["price_in_per_1k"]  = df["model"].map(lambda m: prices_per_1k[m]["in"])
df["price_out_per_1k"] = df["model"].map(lambda m: prices_per_1k[m]["out"])

# Compute the cost of every request — vectorised arithmetic, no loop needed
df["cost_usd"] = (df["tokens_in"]  / 1000 * df["price_in_per_1k"] +
                  df["tokens_out"] / 1000 * df["price_out_per_1k"])

# A boolean column — was this a slow call?
df["was_slow"] = df["latency_ms"] > 2000

# A categorical column with pd.cut — three latency bands
df["latency_band"] = pd.cut(
    df["latency_ms"],
    bins=[0, 1500, 2500, np.inf],
    labels=["fast", "medium", "slow"],
)

df


> 🎯 **Vectorised vs loops.** Notice we did *not* write a `for` loop to compute `cost_usd`. Pandas computes the formula for every row at once. This is faster *and* shorter than a loop — it's why you reach for pandas in the first place.

## 8. Summary statistics & `value_counts`

In [ ]:
print("Latency stats (ms):")
print(f"  mean   : {df['latency_ms'].mean():.0f}")
print(f"  median : {df['latency_ms'].median():.0f}")
print(f"  std    : {df['latency_ms'].std():.0f}")
print(f"  min    : {df['latency_ms'].min()}")
print(f"  max    : {df['latency_ms'].max()}")

print("\nRequests per model:")
print(df["model"].value_counts())

print("\nRequests per customer segment:")
print(df["customer_seg"].value_counts())


## 9. `groupby` — the most useful pandas verb

`groupby` answers questions of the form *"what's the *something* of *something_else*, broken down by *category*?"*. For instance: *"average cost per request, by model"*, *"total tokens, by customer segment"*, *"slow-call rate by model"*.

The mental model is **split → apply → combine**:

```
                    apply (mean, sum, count, …)
                       ↓
   ┌─ gpt-4o-mini : ████  →  $0.0014 / call
df ┤
   └─ claude-haiku: ███   →  $0.0021 / call
                                 ↑
                              combine
```

In [ ]:
# Average cost per request, per model
print("Mean cost per request, by model:")
print(df.groupby("model")["cost_usd"].mean().round(6))

# Multiple aggregations at once, by model
agg = df.groupby("model").agg(
    n_calls         =("request_id", "count"),
    total_tokens    =("tokens_in",  "sum"),
    mean_latency_ms =("latency_ms", "mean"),
    total_cost      =("cost_usd",   "sum"),
).round(4)
agg


In [ ]:
# Grouping by two columns — model and customer segment
cross = df.groupby(["model", "customer_seg"]).agg(
    n          =("request_id", "count"),
    mean_cost  =("cost_usd",   "mean"),
).round(5)
cross


## 10. Reading and writing CSV files

This is how every real project starts and ends. Pandas reads dozens of file formats (CSV, Excel, JSON, Parquet, SQL, …) with a one-line call.

In [ ]:
# Reading from a string is great for self-contained demos.
# In a real notebook this would be:  api_log = pd.read_csv("api_log.csv")

from io import StringIO

csv_text = '''request_id,model,tokens_in,tokens_out,latency_ms,customer_seg
req_009,gpt-4o-mini,520,140,1900,SMB
req_010,claude-haiku,810,210,2380,Enterprise
req_011,gpt-4o-mini,295,82,1420,SMB
req_012,gpt-4o-mini,1050,260,3050,Enterprise
req_013,claude-haiku,640,170,2210,Mid-market
req_014,gpt-4o-mini,470,128,1810,SMB
req_015,claude-haiku,920,235,2670,Enterprise
'''

api_log = pd.read_csv(StringIO(csv_text))
print(api_log)
print(f"\nShape: {api_log.shape}")


In [ ]:
# Group-by on the loaded data: tokens per segment
tokens_by_seg = api_log.groupby("customer_seg")[["tokens_in", "tokens_out"]].sum()
print(tokens_by_seg)

# To save: api_log.to_csv("api_log.csv", index=False)


## 11. A first plot — `df.plot()`

Pandas has a built-in `.plot()` method that wraps matplotlib. It's perfect for quick exploration. The full course does plotting properly in NB 9 (Matplotlib); for the fast track this shortcut is all you need. A glimpse:

In [ ]:
import matplotlib.pyplot as plt

# Total cost per model — a quick bar chart
cost_by_model = df.groupby("model")["cost_usd"].sum().sort_values(ascending=False)

ax = cost_by_model.plot(
    kind="bar",
    color=["#4C72B0", "#DD8452"],
    title="Total cost by model",
    figsize=(7, 4),
    rot=0,
    edgecolor="black",
)
ax.set_ylabel("Total cost (USD)")
ax.grid(axis="y", alpha=0.3)
for x, v in enumerate(cost_by_model.values):
    ax.text(x, v, f"${v:.4f}", ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Latency distribution per model — boxplot
fig, ax = plt.subplots(figsize=(7, 4))
df.boxplot(column="latency_ms", by="model", ax=ax)
ax.set_title("Latency distribution per model")
ax.set_ylabel("Latency (ms)")
plt.suptitle("")     # suppress the auto-title pandas adds
plt.tight_layout()
plt.show()


## 12. The pandas patterns you'll see everywhere

Burn these into memory — they form the daily-use vocabulary of every data scientist:

```python
# Loading
df = pd.read_csv("data.csv")

# Exploring
df.head()                       # see the first rows
df.shape                        # rows, cols
df.describe()                   # summary stats
df.info()                       # types and missing counts

# Selecting
df["col"]                       # one column → Series
df[["col1", "col2"]]            # several columns → DataFrame
df.loc[label, "col"]            # row by label, column by name
df.iloc[0, 0]                   # row by position, column by position

# Filtering
df[df["col"] > 100]
df[(df["col1"] > 100) & (df["col2"] == "x")]

# Adding columns
df["new"] = df["a"] * df["b"]
df["band"] = pd.cut(df["score"], bins=..., labels=...)

# Aggregating
df.groupby("category")["value"].mean()
df.groupby("category").agg(total=("value", "sum"))

# Quick visual
df["amount"].plot(kind="hist", bins=30)
```

Once these feel automatic, the rest of pandas is "more of the same".

## 🧪 Practice exercises

For the next exercises we'll use a slightly larger synthetic dataset: 50 API calls across models, segments, and quarters.

In [ ]:
rng = np.random.default_rng(42)
n = 50

api_data = pd.DataFrame({
    "request_id": [f"req_{i:03d}" for i in range(1, n + 1)],
    "model":      rng.choice(["gpt-4o-mini", "claude-haiku", "gpt-4o"], n),
    "segment":    rng.choice(["SMB", "Mid-market", "Enterprise"], n),
    "quarter":    rng.choice(["Q1", "Q2", "Q3", "Q4"], n),
    "tokens":     rng.integers(120, 2000, n).astype(int),
    "latency_ms": rng.integers(800, 4500, n),
    "user_age":   rng.integers(20, 65, n),
})
api_data.head()


### Exercise 1 — ⭐ Basic exploration

1. Print the **shape** and **dtypes** of `api_data`.
2. Print the **mean** and **median** of `tokens`.
3. Show how many requests there were per `model`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
print(f"shape  = {api_data.shape}")
print(f"\ndtypes:\n{api_data.dtypes}")

print(f"\nmean tokens   = {api_data['tokens'].mean():.1f}")
print(f"median tokens = {api_data['tokens'].median():.1f}")

print("\nRequests per model:")
print(api_data['model'].value_counts())
```
</details>

### Exercise 2 — ⭐⭐ Filtering

Find all requests where **`tokens >= 1500` AND `latency_ms >= 3000`** (the expensive, slow ones). How many are there, and what's their mean latency?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
expensive_slow = api_data[(api_data["tokens"] >= 1500) & (api_data["latency_ms"] >= 3000)]

print(f"Matching rows : {len(expensive_slow)}")
print(f"Mean latency  : {expensive_slow['latency_ms'].mean():.0f} ms")
expensive_slow.head()
```

These are the calls worth profiling first — biggest cost *and* biggest user-experience hit.
</details>

### Exercise 3 — ⭐⭐ Group-by

For each `model`, compute:

1. Total `tokens`.
2. Number of requests.
3. The **segment** that consumed the most tokens for that model (think: groupby on two columns).

Tip: `df.groupby(["model", "segment"])["tokens"].sum()` returns a stacked Series — combine with `.idxmax()` or `.sort_values()`.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
# 1 & 2: per-model summary
summary = api_data.groupby("model").agg(
    total_tokens =("tokens", "sum"),
    n_requests   =("tokens", "count"),
)
print(summary)

# 3: top segment per model
grouped = api_data.groupby(["model", "segment"])["tokens"].sum()
print("\nTop segment per model:")
print(grouped.groupby(level=0).idxmax())   # → (model, segment) tuples
```

The pattern `grouped.groupby(level=0).idxmax()` finds, *within each model group*, which segment had the highest total tokens — a one-line "argmax over a subgroup" trick worth remembering.
</details>

### Exercise 4 — ⭐⭐ Visual exploration

Create two plots:

1. A **bar chart** of total `tokens` per `model`.
2. A **histogram** of `latency_ms` with 10 bins.

Add titles and axis labels.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1) Bar chart
api_data.groupby("model")["tokens"].sum().plot(
    kind="bar", ax=axes[0], color="#4C72B0", edgecolor="black"
)
axes[0].set_title("Total tokens per model")
axes[0].set_ylabel("Tokens")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(axis="y", alpha=0.3)

# 2) Histogram
api_data["latency_ms"].plot(
    kind="hist", bins=10, ax=axes[1], color="#DD8452", edgecolor="black"
)
axes[1].set_title("Latency distribution")
axes[1].set_xlabel("Latency (ms)")

plt.tight_layout()
plt.show()
```
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞

The cell below should print the **average latency per model** but it raises an error. Fix it.

In [ ]:
# 👇 Your fixed/corrected version goes here — write or paste it below.
# Buggy:
# print(api_data.groupby(model).mean(latency_ms))


<details>
<summary>💡 <b>Solution</b></summary>

Two issues:

1. `model` and `latency_ms` need to be **strings** (column names), not bare identifiers.
2. `.mean(latency_ms)` is not how you pick a column — `.mean()` takes no positional column name.

```python
print(api_data.groupby("model")["latency_ms"].mean().round(0))
```

Or, using `.agg`:

```python
print(api_data.groupby("model").agg(avg_latency=("latency_ms", "mean")).round(0))
```

Both work. The `.agg(...)` form is preferred once you want multiple aggregations at once.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise C — ⭐⭐⭐ Pivot to a region × month revenue table

You have a long-format DataFrame:

```python
import pandas as pd
df = pd.DataFrame({
    "month":   ["Jan", "Jan", "Feb", "Feb", "Mar", "Mar"],
    "region":  ["EU",  "US",  "EU",  "US",  "EU",  "US"],
    "revenue":[110,    180,    130,    160,    140,    175],
})
```

Produce a **wide-format** table with months as rows and regions as columns, plus a `Total` column summing across regions.

In [ ]:
# Your code here  👇
import pandas as pd
df = pd.DataFrame({
    "month":   ["Jan", "Jan", "Feb", "Feb", "Mar", "Mar"],
    "region":  ["EU",  "US",  "EU",  "US",  "EU",  "US"],
    "revenue":[110,    180,    130,    160,    140,    175],
})

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import pandas as pd
df = pd.DataFrame({
    "month":   ["Jan", "Jan", "Feb", "Feb", "Mar", "Mar"],
    "region":  ["EU",  "US",  "EU",  "US",  "EU",  "US"],
    "revenue":[110,    180,    130,    160,    140,    175],
})

wide = df.pivot(index="month", columns="region", values="revenue")
wide["Total"] = wide.sum(axis=1)
print(wide)
```

**Reasoning.** `pivot` is the textbook tool for long-to-wide reshaping. Three things worth noting. (1) The arguments form a sentence: "each `index` value becomes a row, each `columns` value becomes a column, populate with `values`." (2) The result is indexed by `month`, which may or may not preserve calendar order — set it with `wide.reindex(["Jan","Feb","Mar"])` if you need a specific order. (3) `wide.sum(axis=1)` sums along rows (each row's two region values) — `axis=0` would sum each column. Choosing the right axis is one of pandas's most common off-by-one mistakes, so always read it aloud: *"sum across the rows, leaving one number per row."*
</details>

### Stretch exercise D — ⭐⭐⭐ IQR-based outlier detection per group

Given the DataFrame below, flag rows where `value` is more than 1.5 × IQR outside its group's quartile range — i.e. apply the classic boxplot rule **separately per group**.

```python
df = pd.DataFrame({
    "group": ["A"]*8 + ["B"]*8,
    "value": [10, 12, 11, 13, 10, 14, 12, 90,
              50, 52, 49, 51, 53, 50, 1, 52],
})
```

Add an `is_outlier` boolean column. Expected: row 7 (value 90 in A) and row 14 (value 1 in B) are flagged.

In [ ]:
# Your code here  👇
import pandas as pd
df = pd.DataFrame({
    "group": ["A"]*8 + ["B"]*8,
    "value": [10, 12, 11, 13, 10, 14, 12, 90,
              50, 52, 49, 51, 53, 50, 1, 52],
})

# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import pandas as pd
df = pd.DataFrame({
    "group": ["A"]*8 + ["B"]*8,
    "value": [10, 12, 11, 13, 10, 14, 12, 90,
              50, 52, 49, 51, 53, 50, 1, 52],
})

def flag_outliers(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    return (s < lo) | (s > hi)

df["is_outlier"] = df.groupby("group")["value"].transform(flag_outliers)
print(df)
```

**Reasoning.** Two important pandas patterns at once. (1) The boxplot rule — `Q1 - 1.5·IQR` and `Q3 + 1.5·IQR` as outlier fences — is robust to skewed data because it uses quantiles, not means. It's the rule matplotlib's `.boxplot` whiskers use by default. (2) `groupby(...).transform(fn)` is the right tool whenever you want **per-group computation back at row level** (same number of rows in, same number out). Compare with `.apply` (can return any shape) and `.agg` (collapses each group to one row). Using `transform` lets us assign the result back as a column in one line.
</details>

## 🧠 Key takeaways

1. A **DataFrame** is a labelled table; a **Series** is one column.
2. Use `df.head() / .shape / .dtypes / .info() / .describe()` to inspect a new dataset.
3. Select columns with `df["col"]` or `df[[col1, col2]]`; rows with `.loc[label]` or `.iloc[position]`.
4. **Boolean masks** drive filtering: `df[mask]`. Combine masks with `&`, `|`, `~`, and wrap each condition in parentheses.
5. **`groupby` → aggregation** is the workhorse of analysis (`mean`, `sum`, `count`, custom funcs).
6. **Derived columns** turn raw data into KPIs — vectorised arithmetic, no loops.
7. `df.plot()` gets you to a chart in one line.
8. `pd.read_csv` / `df.to_csv` is the universal I/O — almost every project starts and ends there.

## ✅ Self-assessment

- [ ] Build a DataFrame from a dict-of-lists
- [ ] Inspect a new dataset with `head()`, `shape`, `dtypes`, `info()`, `describe()`
- [ ] Select columns and rows with `.loc` and `.iloc`
- [ ] Filter rows with a boolean mask combining multiple conditions
- [ ] Add a derived column from a formula
- [ ] Use `groupby(...).agg(...)` to compute multiple aggregates per group
- [ ] Read a CSV and write one back

## 🚀 Next step

Continue with **Notebook 7 (fast track) — Visualization & Statistics**, where you'll turn these DataFrames into charts that make the pattern obvious and statistics that tell you whether the pattern is real.